# Structured text values - Rust

All 12 Rust examples from [docs/core/text.md](https://platob.github.io/yggdryl/core/text/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::Value;
use yggdryl::text::{Json, TextCodec};

let quote = Json.loads(r#"{"symbol":"AAPL","price":12.5}"#)?;

assert_eq!(quote.path("symbol").and_then(Value::as_str), Some("AAPL"));
assert_eq!(quote.keys(), vec!["symbol", "price"]);
assert_eq!(Json.dumps(&quote)?, r#"{"symbol":"AAPL","price":12.5}"#);

## What a value can be

In [ ]:
use yggdryl::Value;

assert!(Value::Null.is_null());
assert_eq!(Value::from("AAPL").kind(), "string");
assert!(Value::from(1.5).is_number());

// Width is representation, not identity.
assert_eq!(Value::I64(1), Value::U64(1));

// Narrowing refuses to lose magnitude rather than wrapping.
assert_eq!(Value::from(7_i64).as_i64(), Some(7));
assert_eq!(Value::from(i128::MAX).as_i64(), None);

// Floats keep their exact bits: signed zeros stay apart, every NaN is one value.
assert_ne!(Value::from(-0.0), Value::from(0.0));
assert_eq!(Value::from(f64::NAN), Value::from(f64::NAN));

## Reading a shape you do not control

In [ ]:
use yggdryl::Value;
use yggdryl::text::{Json, TextCodec};

let order = Json.loads(r#"{"symbol":"AAPL","legs":[{"price":12},{"price":13}],"venue":null}"#)?;

// One dotted path walks mapping keys and sequence indexes.
assert_eq!(order.path("legs.1.price").and_then(Value::as_i64), Some(13));

// A segment that does not resolve is absence, not an error.
assert!(order.path("legs.9.price").is_none());
assert!(order.path("symbol.price").is_none());

assert_eq!(order.keys(), vec!["symbol", "legs", "venue"]);
assert!(order.contains_key("venue"));
assert_eq!(order.entries().count(), 3);

// A present null counts as absent for a default.
let fallback = Value::from("XPAR");
assert_eq!(order.get_or("venue", &fallback), &fallback);

## Rebuilding a mapping

In [ ]:
use yggdryl::Value;
use yggdryl::text::{Json, TextCodec};

let order = Json.loads(r#"{"symbol":"AAPL","venue":null}"#)?;

// Replacing keeps position; a new key is appended.
let updated = order.with_key("venue", "XPAR")?.with_key("currency", "EUR")?;
assert_eq!(updated.keys(), vec!["symbol", "venue", "currency"]);
assert_eq!(updated.path("venue").and_then(Value::as_str), Some("XPAR"));

let trimmed = updated.without_key("venue")?;
assert_eq!(trimmed.keys(), vec!["symbol", "currency"]);

// Removing something absent changes nothing.
assert_eq!(trimmed.without_key("absent")?, trimmed);

// Rebuilding something that is not a mapping says what it is.
let message = Value::from("AAPL")
    .with_key("symbol", "AAPL")
    .unwrap_err()
    .to_string();
assert!(message.contains("expected a mapping"), "{message}");
assert!(message.contains("string"), "{message}");

## A name is not a type

In [ ]:
use yggdryl::{Value, json, yaml};

// `type: "tag"` once named a carrier that held a free-form name over an
// untyped payload. It names nothing now, so it is not an envelope and the
// document is the mapping its syntax always was.
let source = br#"{"$yggdryl":{"version":1,"type":"tag","tag":"app:Trade","value":{"symbol":"AAPL"}}}"#;
let decoded = json::from_slice(source)?;
assert_eq!(
    decoded.path("$yggdryl.tag").and_then(Value::as_str),
    Some("app:Trade")
);
// ... and it goes back out unchanged, escaped through the mapping envelope.
assert_eq!(json::from_slice(&json::to_vec(&decoded)?)?, decoded);

// A YAML application tag is the annotation YAML defines it to be, so the
// node under it decodes as the plain value it annotates.
let annotated = yaml::from_slice(b"!app:Trade {symbol: AAPL}\n")?;
assert_eq!(annotated.path("symbol").and_then(Value::as_str), Some("AAPL"));

## One value against one datatype

In [ ]:
use yggdryl::{DataType, TypedValue, Value};

let price = TypedValue::from_parts(DataType::Int64, Value::from(7_i64))?;
assert_eq!(price.data_type(), &DataType::Int64);
assert_eq!(price.value(), &Value::I64(7));

// The value is checked against the datatype, through the same walk a column
// value takes, so a pairing that exists is one that holds.
assert!(TypedValue::from_parts(DataType::Int64, Value::from("seven")).is_err());

// A value can also name its own datatype.
assert_eq!(
    TypedValue::from_value(Value::from(1.5))?.data_type(),
    &DataType::Float64
);

// A null is accepted by every datatype: nullability belongs to the field that
// holds the column, not to the value in it.
let missing = TypedValue::from_parts(DataType::Int64, Value::Null)?;
assert!(missing.is_null());
assert!(!price.is_null());

## A typed value per datatype

In [ ]:
use yggdryl::generic::{Int64Value, TimestampValue, Utf8Value};
use yggdryl::{DataType, TimeUnit, TypedValue, Value};

// A statically known datatype needs only its value.
let price = Int64Value::new(Value::from(7_i64))?;
assert_eq!(price.data_type(), &DataType::Int64);

// The marker is checked, and so is the value against it.
assert!(Int64Value::new(Value::from("seven")).is_err());
assert!(Int64Value::try_from_parts(DataType::Utf8, Value::from("seven")).is_err());
assert_eq!(Utf8Value::try_from_value(Value::from("AAPL"))?.value(), &Value::from("AAPL"));

// A parameterized datatype keeps its parameters in the pairing, not the marker.
let at = TimestampValue::try_from_parts(
    DataType::Timestamp(TimeUnit::Microsecond, None),
    Value::timestamp(0, TimeUnit::Microsecond, None)?,
)?;
assert_eq!(at.data_type(), &DataType::Timestamp(TimeUnit::Microsecond, None));

// Narrowing and widening move the same two halves between markers.
let dynamic: TypedValue = at.into_any();
assert!(dynamic.try_into_typed::<yggdryl::field::binary::Utf8>().is_err());

## Four formats, one surface

In [ ]:
use yggdryl::MimeType;
use yggdryl::text::{Json, Jsonl, TextCodec, Toml, Yaml};

let quote = Json.loads(r#"{"symbol":"AAPL"}"#)?;

// One value, four grammars, one set of methods.
assert_eq!(Json.dumps(&quote)?, r#"{"symbol":"AAPL"}"#);
assert_eq!(Toml.loads(&Toml.dumps(&quote)?)?, quote);
assert_eq!(Yaml.loads(&Yaml.dumps(&quote)?)?, quote);

assert_eq!(Json.mime_type(), MimeType::JSON);
assert_eq!(Jsonl.mime_type(), MimeType::JSON_LINES);
assert!(Jsonl.is_multi_document());
assert!(!Toml.is_multi_document());

// Bytes and readers are the same operation on a different carrier.
let bytes = Json.dump_vec(&quote)?;
assert_eq!(Json.load_slice(&bytes)?, quote);
assert_eq!(Json.read(bytes.as_slice())?, quote);

let rows = Jsonl.loads_all("{\"id\":1}\n{\"id\":2}\n")?;
assert_eq!(rows.len(), 2);

## Inferring the format

In [ ]:
use yggdryl::text::{from_str_inferred, infer_format};
use yggdryl::{Format, Value};

// Valid JSON wins, because most JSON is also valid YAML.
assert_eq!(infer_format(br#"{"symbol":"AAPL"}"#)?, Format::Json);
assert_eq!(infer_format(b"symbol = \"AAPL\"\n")?, Format::Toml);
assert_eq!(infer_format(b"symbol: AAPL\n")?, Format::Yaml);

// Inferring and decoding is one parse, not two.
let (format, value) = from_str_inferred("symbol = \"AAPL\"\n")?;
assert_eq!(format, Format::Toml);
assert_eq!(value.path("symbol").and_then(Value::as_str), Some("AAPL"));

// A name decides too, and JSON Lines needs one.
assert_eq!(Format::from_path("events.jsonl")?, Format::JsonLines);
assert_eq!(Format::from_extension(".YML")?, Format::Yaml);
assert_eq!(Format::from_str("application/toml")?, Format::Toml);

## Bounds on untrusted input

In [ ]:
use yggdryl::Limits;
use yggdryl::text::{Json, TextCodec};

let strict = Json.with_limits(Limits::new(2, 1024, 64, 4));
assert_eq!(strict.limits().max_depth(), 2);

// A root container is depth 1.
assert!(strict.loads(r#"{"a":[1]}"#).is_ok());
assert!(strict.loads(r#"{"a":[[1]]}"#).is_err());

// A bare format value uses the defaults.
assert_eq!(Json.limits(), Limits::default());
assert_eq!(Limits::default().max_depth(), 128);
assert_eq!(Limits::default().max_input_bytes(), 64 * 1024 * 1024);
assert_eq!(Limits::default().max_documents(), 1_024);

## Failures carry a byte position

In [ ]:
use yggdryl::text::{Json, TextCodec, Toml};

let message = Json.loads(r#"{"symbol": "#).unwrap_err().to_string();
assert!(message.contains("invalid json data at byte 11"), "{message}");

let message = Toml.loads("symbol = ").unwrap_err().to_string();
assert!(message.contains("invalid toml data at byte 9"), "{message}");

## Through a storage handle

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::text::{Plan, dump, load};
use yggdryl::{Codec, Format, Url, Value};

let mut handle =
    Buffer::new().with_media_type(Url::from_str("file:///quote.json.gz")?.media_type());

// The name is the whole configuration.
let plan = Plan::infer(&handle)?;
assert_eq!(plan.format(), Format::Json);
assert_eq!(plan.codec(), Codec::Gzip);

let quote = Value::from_mapping([(Value::from("symbol"), Value::from("AAPL"))])?;
dump(&mut handle, &quote)?;

// The stored bytes really are gzip, and reading decompresses them.
assert_eq!(&handle.as_slice()[..2], &[0x1F, 0x8B]);
assert_eq!(load(&handle)?, quote);